<a href="https://colab.research.google.com/github/immaximo/Hybrid-ML-LLM-Tutoring-Model-for-Cloud-Certification-Preparation-/blob/main/Hybrid_en_GNNKT2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
torch.cuda.is_available()

False

In [ ]:
import torch
!pip install torch torchvision

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install torch_geometric

In [ ]:
import pandas as pd
import numpy as np
import torch

from torch_geometric.data import Data

# Load datasets here
interactions = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/synchronized_interaction_logs.csv")
questions = pd.read_excel("/content/drive/MyDrive/Colab Notebooks/CDL_150_Questions_With_ID.xlsx")
googlequestions = pd.read_json("/content/drive/MyDrive/Colab Notebooks/google_cloud_quiz_eval.json")
studentquiz = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/student_quiz_dataset.csv")

# For matching data
questions = questions.rename(columns={'Question_ID': 'question_id'})
questions['skill_id'] = questions['Module']
interactions = interactions.rename(columns={'student_id': 'user_id'})
studentquiz = studentquiz.rename(columns={'student_id': 'user_id'})


interactions = interactions.merge(studentquiz[['user_id', 'question_id', 'status']],
                                  on=['user_id', 'question_id'],
                                  how='left')


interactions['correct'] = interactions['status'].map({'correct': 1, 'incorrect': 0}).fillna(0).astype(int)

# Drop the original status column as it's no longer needed
interactions = interactions.drop(columns=['status'], errors='ignore')

# Merge interaction and skills
data = interactions.merge(questions, on="question_id", how="inner")
data = data.sort_values(by=["user_id", "timestamp"])

# IDs
user2idx = {u: i for i, u in enumerate(data["user_id"].unique())}
q2idx = {q: i for i, q in enumerate(data["question_id"].unique())}
s2idx = {s: i for i, s in enumerate(data["skill_id"].unique())}

data["user_id"] = data["user_id"].map(user2idx)
data["question_id"] = data["question_id"].map(q2idx)
data["skill_id"] = data["skill_id"].map(s2idx)

num_users = len(user2idx)
num_skills = len(s2idx)

# Student Sequences
student_sequences = []
grouped = data.groupby("user_id")

for user, group in grouped:
    student_sequences.append({
        "student_id": user,
        "q_seq": torch.tensor(group["question_id"].values, dtype=torch.long),
        "s_seq": torch.tensor(group["skill_id"].values, dtype=torch.long),
        "r_seq": torch.tensor(group["correct"].values, dtype=torch.float)
    })

# Skill graph
adj_matrix = np.zeros((num_skills, num_skills))
for seq in student_sequences:
    skills = seq["s_seq"].numpy()
    for i in range(len(skills)-1):
        s1, s2 = skills[i], skills[i+1]
        adj_matrix[s1, s2] += 1
        adj_matrix[s2, s1] += 1

edges = np.array(np.nonzero(adj_matrix))
edge_index = torch.tensor(edges, dtype=torch.long)
edge_weight = torch.tensor(adj_matrix[edges[0], edges[1]], dtype=torch.float)

x = torch.eye(num_skills, dtype=torch.float)

# Create PyG data object
data_gnn = Data(x=x, edge_index=edge_index, edge_attr=edge_weight)
data_gnn.student_sequences = student_sequences

# Save
torch.save(data_gnn, "kt_all_models_prepost.pt")
print("Preprocessing complete. Dataset ready for all models (AKT, DKT, SAKT, GRKT/GNN).")

Preprocessing complete. Dataset ready for all models (AKT, DKT, SAKT, GRKT/GNN).


<h1>GNN MODEL</H1>

In [ ]:
import torch.nn.functional as F
from torch_geometric.nn import GCNConv

class GNNKTModel(nn.Module):
    def __init__(self, num_skills, hidden_dim, output_dim, n_layers=1):
        super(GNNKTModel, self).__init__()
        self.num_skills = num_skills
        self.hidden_dim = hidden_dim
        self.output_dim = output_dim
        self.n_layers = n_layers

        # GNN for skill embedding (using GCNConv)
        # Input to GCN is a one-hot encoding of skills
        self.conv1 = GCNConv(num_skills, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim) # Additional layer

        # GRU for sequential modeling of student interactions
        # Input to GRU will be concatenation of question embedding (skill embedding) and correctness
        # So, input_size for GRU will be hidden_dim (for skill embedding) + 1 (for correctness)
        self.gru = nn.GRU(input_size=hidden_dim + 1, hidden_size=hidden_dim, num_layers=n_layers, batch_first=True)

        # Output layer to predict correctness
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, s_seq, r_seq, edge_index, edge_attr, x_skills):
        # 1. Generate skill embeddings using GNN
        # x_skills are the one-hot encoded skill features (num_skills x num_skills identity matrix)
        skill_embeddings = F.relu(self.conv1(x_skills, edge_index, edge_attr))
        skill_embeddings = F.relu(self.conv2(skill_embeddings, edge_index, edge_attr))

        # 2. Prepare input for GRU
        # For each interaction in the sequence, get its corresponding skill embedding
        # s_seq contains skill indices
        current_skill_embeddings = skill_embeddings[s_seq] # Shape: (batch_size, seq_len, hidden_dim)

        # Concatenate skill embeddings with past response (r_seq)
        # r_seq needs to be unsqueezed to match dimensions
        gru_input = torch.cat((current_skill_embeddings, r_seq.unsqueeze(-1)), dim=-1) # Shape: (batch_size, seq_len, hidden_dim + 1)

        # 3. Process sequence with GRU
        gru_output, _ = self.gru(gru_input)

        # 4. Predict correctness for the next interaction
        # We are predicting the correctness of the *next* interaction, so shift the target
        # The GRU output at step t predicts the outcome at step t+1
        # For training, we usually predict all steps and compare with shifted ground truth.
        predictions = self.fc(gru_output) # Shape: (batch_size, seq_len, output_dim)

        return predictions.squeeze(-1) # Squeeze output_dim if it's 1

print("GNNKTModel architecture defined.")

GNNKTModel architecture defined.


<H1>TRAINING AND ACCURACY EVALUATION

In [ ]:
!pip install optuna


In [ ]:
!pip install pytorch_optimizer

In [ ]:
import optuna
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, mean_squared_error
from pytorch_optimizer import SAM

def objective(trial):
    # Hyperparameters
    hidden_dim = trial.suggest_categorical('hidden_dim', [32, 64, 128])
    learning_rate = trial.suggest_float('learning_rate', 1e-5, 1e-2, log=True)
    n_layers = trial.suggest_int('n_layers', 1, 3)
    rho = trial.suggest_float('rho', 0.01, 0.2, log=True)
    n_epochs = 5

    output_dim = 1

    model = GNNKTModel(num_skills, hidden_dim, output_dim, n_layers)

    base_optimizer = optim.Adam
    optimizer = SAM(model.parameters(), base_optimizer, lr=learning_rate, rho=rho)
    criterion = nn.BCEWithLogitsLoss()

    train_sequences, test_sequences = train_test_split(data_gnn.student_sequences, test_size=0.2, random_state=42)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    edge_index = data_gnn.edge_index.to(device)
    edges_attr = data_gnn.edge_attr.to(device)
    x_skills = data_gnn.x.to(device)

    print(f"\n--- Starting Trial {trial.number} ---")
    print(f"Params: hidden_dim={hidden_dim}, lr={learning_rate:.6f}, n_layers={n_layers}, rho={rho:.4f}")

    # SAM
    for epoch in range(n_epochs):
        model.train()
        total_loss = 0
        for student_data in train_sequences:
            q_seq = student_data['q_seq'].to(device)
            s_seq = student_data['s_seq'].to(device)
            r_seq = student_data['r_seq'].to(device)

            if len(s_seq) > 1:
                predictions = model(s_seq[:-1].unsqueeze(0), r_seq[:-1].unsqueeze(0), edge_index, edges_attr, x_skills)
                target = r_seq[1:].unsqueeze(0)
                loss = criterion(predictions, target)
                loss.backward()
                optimizer.first_step(zero_grad=True)

                predictions_second = model(s_seq[:-1].unsqueeze(0), r_seq[:-1].unsqueeze(0), edge_index, edges_attr, x_skills)
                loss_second = criterion(predictions_second, target)
                loss_second.backward()
                optimizer.second_step(zero_grad=True)

                total_loss += loss.item()
        print(f"Epoch {epoch+1}/{n_epochs}, Loss: {total_loss/len(train_sequences):.4f}")

    # Evaluation
    print(f"Evaluating Trial {trial.number} on test set...")
    model.eval()
    all_predictions = []
    all_targets = []

    with torch.no_grad():
        for student_data in test_sequences:
            q_seq = student_data['q_seq'].to(device)
            s_seq = student_data['s_seq'].to(device)
            r_seq = student_data['r_seq'].to(device)

            if len(s_seq) > 1:
                predictions = model(s_seq[:-1].unsqueeze(0), r_seq[:-1].unsqueeze(0), edge_index, edges_attr, x_skills)
                target = r_seq[1:].unsqueeze(0)

                probabilities = torch.sigmoid(predictions)

                all_predictions.extend(probabilities.cpu().numpy().flatten().tolist())
                all_targets.extend(target.cpu().numpy().flatten().tolist())

    # Calculate AUC and RMSE
    if len(all_targets) > 0:
        auc = roc_auc_score(all_targets, all_predictions)
        rmse = torch.sqrt(torch.tensor(mean_squared_error(all_targets, all_predictions), dtype=torch.float32)).item()
        print(f"=> Trial {trial.number} finished. AUC: {auc:.4f}, RMSE: {rmse:.4f}")
        return auc
    else:
        print(f"=> Trial {trial.number} finished. No sufficient data for evaluation.")
        return 0.0

In [ ]:
import optuna

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=5)

print("Number of finished trials: ", len(study.trials))
print("Best trial:")

trial = study.best_trial

print("  Value: ", trial.value)
print("  Params: ")
for key, value in trial.params.items():
    print("    {}: {}".format(key, value))

[I 2026-03-30 11:36:14,360] A new study created in memory with name: no-name-36c72ae5-7bae-4b20-87d6-48b69dcf42fa



--- Starting Trial 0 ---
Params: hidden_dim=128, lr=0.000665, n_layers=2, rho=0.1964
Epoch 1/5, Loss: 0.5750
Epoch 2/5, Loss: 0.5601
Epoch 3/5, Loss: 0.5572
Epoch 4/5, Loss: 0.5469
Epoch 5/5, Loss: 0.5343
Evaluating Trial 0 on test set...


[I 2026-03-30 12:02:50,401] Trial 0 finished with value: 0.7613737622202716 and parameters: {'hidden_dim': 128, 'learning_rate': 0.0006652900818768577, 'n_layers': 2, 'rho': 0.19642978039866743}. Best is trial 0 with value: 0.7613737622202716.


=> Trial 0 finished. AUC: 0.7614, RMSE: 0.4102

--- Starting Trial 1 ---
Params: hidden_dim=128, lr=0.008227, n_layers=2, rho=0.0533
Epoch 1/5, Loss: 0.5670
Epoch 2/5, Loss: 0.5593
Epoch 3/5, Loss: 0.5507
Epoch 4/5, Loss: 0.5383
Epoch 5/5, Loss: 0.5277
Evaluating Trial 1 on test set...


[I 2026-03-30 12:29:12,825] Trial 1 finished with value: 0.770786331373707 and parameters: {'hidden_dim': 128, 'learning_rate': 0.008226738889727426, 'n_layers': 2, 'rho': 0.05334076293097795}. Best is trial 1 with value: 0.770786331373707.


...

=> Trial 2 finished. AUC: 0.6609, RMSE: 0.4385

--- Starting Trial 3 ---
Params: hidden_dim=128, lr=0.000017, n_layers=1, rho=0.0692
Epoch 1/5, Loss: 0.5975
Epoch 2/5, Loss: 0.5839
Epoch 3/5, Loss: 0.5814
Epoch 4/5, Loss: 0.5797
Epoch 5/5, Loss: 0.5784
Evaluating Trial 3 on test set...


[I 2026-03-30 13:04:38,025] Trial 3 finished with value: 0.6057068200892739 and parameters: {'hidden_dim': 128, 'learning_rate': 1.683753813000264e-05, 'n_layers': 1, 'rho': 0.06920198326500689}. Best is trial 1 with value: 0.770786331373707.


=> Trial 3 finished. AUC: 0.6057, RMSE: 0.4484

--- Starting Trial 4 ---
Params: hidden_dim=64, lr=0.000030, n_layers=3, rho=0.0718
Epoch 1/5, Loss: 0.5961
Epoch 2/5, Loss: 0.5854
Epoch 3/5, Loss: 0.5822
Epoch 4/5, Loss: 0.5798
Epoch 5/5, Loss: 0.5776
Evaluating Trial 4 on test set...


[I 2026-03-30 13:36:55,711] Trial 4 finished with value: 0.6119133286413451 and parameters: {'hidden_dim': 64, 'learning_rate': 3.0232556117612367e-05, 'n_layers': 3, 'rho': 0.07176228634130083}. Best is trial 1 with value: 0.770786331373707.


=> Trial 4 finished. AUC: 0.6119, RMSE: 0.4478
Number of finished trials:  5
Best trial:
  Value:  0.770786331373707
  Params: 
    hidden_dim: 128
    learning_rate: 0.008226738889727426
    n_layers: 2
    rho: 0.05334076293097795


<h1>LATENCY

In [ ]:
import time
import torch
import pandas as pd # Ensure pandas is imported

print("Starting interactive latency test (3 runs)...\n")

for run in range(3):
    try:
        user_input = input(f"Run {run + 1}/3 - Please enter a Question ID: ")
        q_id_orig = int(user_input)
    except ValueError:
        print("Invalid input. Please enter a valid numerical Question ID.\n")
        continue

    # Record start time just before fetching the question
    start_time = time.perf_counter()

    # Fetch question details
    q_row = questions[questions['question_id'] == q_id_orig]
    if q_row.empty:
        print(f"Question ID {q_id_orig} not found in the dataset.\n")
        continue

    question_text = q_row['Question'].iloc[0]

    # Record end time immediately after fetching the question
    end_time = time.perf_counter()

    latency_ms = (end_time - start_time) * 1000

    # Print Results
    print(f"\n--- Results ---")
    print(f"Question ID: {q_id_orig}")
    print(f"Question: {question_text}")
    print(f"Latency for question lookup: {latency_ms:.4f} ms\n")

Starting interactive latency test (3 runs)...

Run 1/3 - Please enter a Question ID: 40

--- Results ---
Question ID: 40
Question: What is object storage?
Latency for question lookup: 1.3085 ms

Run 2/3 - Please enter a Question ID: 50

--- Results ---
Question ID: 50
Question: What is AI?
Latency for question lookup: 1.7699 ms

Run 3/3 - Please enter a Question ID: 85

--- Results ---
Question ID: 85
Question: What is an API?
Latency for question lookup: 1.3303 ms



<h1>HYBRIDIZATION


<H2>API

In [ ]:
!pip install openai

import openai
from google.colab import userdata

OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')

client = openai.OpenAI(api_key=OPENAI_API_KEY)

print("OpenAI client initialized.")

OpenAI client initialized.


<h1>RAG

In [ ]:
!pip install sentence-transformers faiss-cpu rank_bm25

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from rank_bm25 import BM25Okapi
import torch
import pandas as pd

questions = pd.read_excel("/content/drive/MyDrive/Colab Notebooks/CDL_150_Questions_With_ID.xlsx")
questions = questions.rename(columns={'Question_ID': 'question_id'})

questions['skill_id'] = questions['Module']

# Dense Retrieval Setup
print("\n--- Setting up Dense Retrieval ---")

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

question_texts = questions['Question'].tolist()
question_embeddings = embedding_model.encode(question_texts, convert_to_tensor=False)

print(f"Generated {len(question_embeddings)} embeddings of dimension {question_embeddings.shape[1]}.")
print(f"First embedding snippet: {question_embeddings[0][:5]}...")

dimension = question_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension) # Using L2 distance for similarity
index.add(question_embeddings.astype('float32')) # FAISS requires float32

print(f"FAISS index created with {index.ntotal} vectors.")

# Sparse Retrieval Setup
print("\n--- Setting up Sparse Retrieval ---")

tokenized_corpus = [doc.lower().split(" ") for doc in question_texts]
bm25 = BM25Okapi(tokenized_corpus)

print("BM25 index created.")


--- Setting up Dense Retrieval ---


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Generated 150 embeddings of dimension 384.
First embedding snippet: [-0.0426912   0.03437556 -0.10072621 -0.09831116 -0.05000662]...
FAISS index created with 150 vectors.

--- Setting up Sparse Retrieval ---
BM25 index created.


In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from rank_bm25 import BM25Okapi

# 1. Load the Curriculum Dataset
curriculum_path = "/content/drive/MyDrive/Colab Notebooks/cdl-TOPIC_DATASET.xlsx"
df_curriculum = pd.read_excel(curriculum_path)

# 2. Prepare text for embedding (combining Topic and Definition)
df_curriculum['rag_text'] = "Topic: " + df_curriculum['Topic'].astype(str) + " | Definition: " + df_curriculum['Definition'].astype(str)
curriculum_texts = df_curriculum['rag_text'].tolist()

# 3. Build Dense Index
print("Building Dense Index for Curriculum...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
curriculum_embeddings = embedding_model.encode(curriculum_texts, convert_to_tensor=False)
dimension = curriculum_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(curriculum_embeddings.astype('float32'))

# 4. Build Sparse Index
print("Building Sparse Index for Curriculum...")
tokenized_corpus = [doc.lower().split(" ") for doc in curriculum_texts]
bm25 = BM25Okapi(tokenized_corpus)

print("Indexes built successfully!\n")

def hybrid_search(query, top_k=5):
    # 1. Dense Retrieval
    query_embedding = embedding_model.encode([query]).astype('float32')
    distances, dense_indices = index.search(query_embedding, top_k)
    dense_results = dense_indices[0].tolist()

    # 2. Sparse Retrieval
    tokenized_query = query.lower().split(" ")
    bm25_scores = bm25.get_scores(tokenized_query)
    sparse_results = np.argsort(bm25_scores)[::-1][:top_k].tolist()

    # 3. RRF Combination
    rrf_scores = {}
    for rank, idx in enumerate(dense_results):
        rrf_scores[idx] = rrf_scores.get(idx, 0.0) + 1.0 / (60 + rank + 1)
    for rank, idx in enumerate(sparse_results):
        rrf_scores[idx] = rrf_scores.get(idx, 0.0) + 1.0 / (60 + rank + 1)

    combined_results = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]
    final_indices = [idx for idx, score in combined_results]

    # 4. Fetch the actual curriculum context
    retrieved_context = df_curriculum.iloc[final_indices].copy()

    # Map columns so the downstream LLM pipeline can read them properly!
    retrieved_context['skill_id'] = retrieved_context['Topic']
    retrieved_context['Question'] = retrieved_context['Topic']
    retrieved_context['Explanation'] = retrieved_context['Definition']

    return retrieved_context

# Let's test the hybrid search with a sample query
test_query = "What is machine learning?"
print(f"Query: {test_query}\n")
results = hybrid_search(test_query, top_k=3)
display(results[['Topic', 'Definition']])

Building Dense Index for Curriculum...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Building Sparse Index for Curriculum...
Indexes built successfully!

Query: What is machine learning?



,Topic,Definition
155,Executing ML ML,The process of running machine learning models.
46,Machine learning (ML),Branch of AI where computers learn from data a...
86,Authorization,Determining what a user is allowed to do.


Check for column names

In [ ]:
import pandas as pd

curriculum_path = "/content/drive/MyDrive/Colab Notebooks/cdl-TOPIC_DATASET.xlsx"

try:
    df_curriculum = pd.read_excel(curriculum_path)
    print("--- Curriculum Dataset Columns ---")
    print(df_curriculum.columns.tolist())
    print("\n--- First 2 rows ---")
    display(df_curriculum.head(2))
except Exception as e:
    print(f"Error loading curriculum: {e}")

--- Curriculum Dataset Columns ---
['ID', 'Course #', 'Module #', 'Topic', 'Definition', 'Examples', 'Related Topics']

--- First 2 rows ---


,ID,Course #,Module #,Topic,Definition,Examples,Related Topics
0,1,Course 1,Module 1,Digital Transformation,When an organization uses new technologies to ...,Modifying business processes; culture; and cus...,Innovation; Cloud Technology
1,2,Course 1,Module 1,Cloud technology,The technology and processes needed to store; ...,Data accessed via internet vs. local hard drive.,Compute power; Computing


High-Scores

In [ ]:
import pandas as pd

# File paths
STUDENT_ANSWERS_CSV_PATH = "/content/drive/MyDrive/Colab Notebooks/high_score_test.csv"
CDL_TOPIC_DATASET_PATH = "/content/drive/MyDrive/Colab Notebooks/cdl-TOPIC_DATASET.xlsx"
QUIZ_ANSWER_KEY_PATH = "/content/drive/MyDrive/Colab Notebooks/quiz_answer_key.csv"

# Column name config
COL_QUESTION_ID = 'question_id'
COL_STUDENT_ANSWER = 'answer'
COL_CORRECT_ANSWER = 'correct_answer_label'

def generate_personalized_study_guide(csv_path, curriculum_path, answer_key_path):
    print(f"Loading student answers from: {csv_path}")
    print(f"Loading answer key from: {answer_key_path}")
    print(f"Loading curriculum dataset from: {curriculum_path}")

    # Load datasets
    try:
        student_answers = pd.read_csv(csv_path)
        answer_key = pd.read_csv(answer_key_path)
        cdl_curriculum = pd.read_excel(curriculum_path)
    except FileNotFoundError as e:
        return f"File not found error: {e}\nPlease check the paths and try again."

    # Check student answers against the answer key
    if COL_QUESTION_ID not in student_answers.columns or COL_QUESTION_ID not in answer_key.columns:
        return f"Error: Both CSVs must contain a '{COL_QUESTION_ID}' column to match them up."

    merged = pd.merge(student_answers, answer_key, on=COL_QUESTION_ID, how='left')

    if COL_STUDENT_ANSWER not in merged.columns or COL_CORRECT_ANSWER not in merged.columns:
        return f"Error: Check column names. Could not find '{COL_STUDENT_ANSWER}' or '{COL_CORRECT_ANSWER}' after merging."

    # Create a boolean correctness column by comparing strings (ignoring case and whitespace)
    merged['is_correct'] = (
        merged[COL_STUDENT_ANSWER].astype(str).str.strip().str.lower() ==
        merged[COL_CORRECT_ANSWER].astype(str).str.strip().str.lower()
    ).astype(int)

    # Filter questions for the study guide
    incorrect_answers = merged[merged['is_correct'] == 0]
    perfect_score = False

    if incorrect_answers.empty:
        print("Perfect score detected! Generating a comprehensive review guide based on all answered questions...")
        target_questions = merged
        perfect_score = True
    else:
        print(f"Found {len(incorrect_answers)} incorrect answers. Gathering context from Hybrid RAG...")
        target_questions = incorrect_answers

    # Gather context using our Hybrid RAG for the targeted topics/questions
    rag_context = ""
    for index, row in target_questions.iterrows():
        # Use the question ID or topic to search our RAG
        search_query = str(row.get('topic', f"Question ID {row.get(COL_QUESTION_ID)} concept"))

        # Fetch top 3 most relevant context blocks to expand the scope
        rag_results = hybrid_search(search_query, top_k=3)

        # Append the retrieved content (including Related Topics) to our context
        for _, q_row in rag_results.iterrows():
            rag_context += f"Topic: {q_row.get('Topic', 'Unknown')}\n"
            rag_context += f"Definition: {q_row.get('Definition', '')}\n"
            # Include related topics from the dataset so the LLM can expand the study guide
            rag_context += f"Related Topics to Mention: {q_row.get('Related Topics', 'None')}\n\n"

    # Inject context into LLM (OpenAI GPT API)
    print("Generating study guide via OpenAI API...")

    if perfect_score:
        prompt_intro = "A student has just taken a practice exam and got a perfect score! Provide an extremely comprehensive, advanced reviewer to reinforce and deepen their knowledge on these topics."
    else:
        prompt_intro = "A student has just taken a practice exam and got several topics wrong. Provide a personalized, extensively detailed reviewer to help them deeply understand their mistakes and master the material."

    prompt = f"""
    You are an expert Google Cloud Digital Leader tutor.
    {prompt_intro}

    CRITICAL INSTRUCTIONS:
    1. CLEAN PLAIN TEXT ONLY: DO NOT use any Markdown formatting. NO asterisks (*), NO hashtags (#), NO underscores (_), and NO bold/italic formatting. Use ALL CAPS for headers, simple numbers (1., 2.), and clean line breaks to structure your output.
    2. BE HIGHLY DESCRIPTIVE & CONTEXTUAL: Provide a massive, extensive, and comprehensive reviewer. Do not give brief summaries. Tie everything back to Google Cloud principles, business values, and real-world environments.
    3. MENTION RELATED TOPICS: In the context below, there are "Related Topics" listed for the main concepts. You MUST weave these related topics into your explanations to expand the length and depth of the study guide. Show how they connect!
    4. NO CRITIQUE: DO NOT critique the quiz itself or mention test format. Focus purely on teaching.

    For every topic the student needs to review, you MUST format the output strictly as follows:

    TOPIC: [TOPIC NAME IN ALL CAPS]

    DESCRIPTION AND IN-DEPTH EXPLANATION:
    (A comprehensive and highly detailed explanation of the concept and its contextual importance in Google Cloud.)

    RELATED CONCEPTS TO KNOW:
    (Explain the related topics provided in the context. Show how they interconnect with the main topic to build a broader understanding.)

    REAL-WORLD EXAMPLES:
    (Detailed, practical examples illustrating exactly how this concept is applied in a business or cloud environment.)

    GUIDE QUESTIONS:
    (3 to 5 thought-provoking questions to thoroughly test their understanding.)

    Context from Curriculum (Including Related Topics):
    {rag_context}

    Please provide the structured, extensive, plain-text reviewer now.
    """

    try:
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": "You are a highly professional, exhaustive, and helpful cloud tutor."},
                {"role": "user", "content": prompt}
            ],
            max_completion_tokens=4000
        )
        llm_response_content = response.choices[0].message.content
        print(f"LLM Response Content Length: {len(llm_response_content)}")
        print(f"LLM Response Content (first 500 chars):\n{llm_response_content[:500]}")
        return llm_response_content
    except Exception as e:
        return f"Error calling OpenAI API: {e}"

# EXECUTE THE PIPELINE

final_study_guide = generate_personalized_study_guide(STUDENT_ANSWERS_CSV_PATH, CDL_TOPIC_DATASET_PATH, QUIZ_ANSWER_KEY_PATH)
print("\n================ GENERATED STUDY GUIDE ================\n")
print(final_study_guide)
print("\n=======================================================\n")

Loading student answers from: /content/drive/MyDrive/Colab Notebooks/high_score_test.csv
Loading answer key from: /content/drive/MyDrive/Colab Notebooks/quiz_answer_key.csv
Loading curriculum dataset from: /content/drive/MyDrive/Colab Notebooks/cdl-TOPIC_DATASET.xlsx
Found 10 incorrect answers. Gathering context from Hybrid RAG...
Generating study guide via OpenAI API...
LLM Response Content Length: 7588
LLM Response Content (first 500 chars):
TOPIC: NEW VALUE VALUE

DESCRIPTION AND IN-DEPTH EXPLANATION:  
In the modern digital world, the explosion of data has led businesses to seek ways to extract meaningful insights to gain a competitive edge. By discovering previously hidden benefits in their data, organizations can unlock new value. This involves a process of transforming raw data into actionable insights that can impact decision-making, customer engagement, and strategy formulation. The goal is to use analytics and data to discov

================ GENERATED STUDY GUIDE ===========

In [ ]:
# Save the deeply generated study guide to a Markdown/Text file in your Drive
output_file_path = "/content/drive/MyDrive/Colab Notebooks/CDL_Personalized_Reviewer.md"

try:
    with open(output_file_path, "w", encoding="utf-8") as f:
        f.write(final_study_guide)
    print(f"✅ Successfully saved the FULL reviewer to:\n{output_file_path}")
    print("You can now open this file in Google Drive to read the complete, untruncated study guide!")
except Exception as e:
    print(f"Error saving file: {e}")

✅ Successfully saved the FULL reviewer to:
/content/drive/MyDrive/Colab Notebooks/CDL_Personalized_Reviewer.md
You can now open this file in Google Drive to read the complete, untruncated study guide!


Checking column names

In [ ]:
import pandas as pd

student_csv = "/content/drive/MyDrive/Colab Notebooks/high_score_test.csv"
answer_key_csv = "/content/drive/MyDrive/Colab Notebooks/quiz_answer_key.csv"

print("--- Inspecting Student Answers CSV ---")
try:
    df_student = pd.read_csv(student_csv)
    print("Columns:", df_student.columns.tolist())
except Exception as e:
    print(f"Error reading student CSV: {e}")

print("\n--- Inspecting Answer Key CSV ---")
try:
    df_key = pd.read_csv(answer_key_csv)
    print("Columns:", df_key.columns.tolist())
except Exception as e:
    print(f"Error reading answer key CSV: {e}")

--- Inspecting Student Answers CSV ---
Columns: ['question_id', 'answer']

--- Inspecting Answer Key CSV ---
Columns: ['question_id', 'correct_answer_label', 'correct_answer_text']


Low-Scores

In [ ]:
import pandas as pd

# File paths
STUDENT_ANSWERS_CSV_PATH = "/content/drive/MyDrive/Colab Notebooks/low_score_test.csv"
CDL_TOPIC_DATASET_PATH = "/content/drive/MyDrive/Colab Notebooks/cdl-TOPIC_DATASET.xlsx"
QUIZ_ANSWER_KEY_PATH = "/content/drive/MyDrive/Colab Notebooks/quiz_answer_key.csv"

# Column name config
COL_QUESTION_ID = 'question_id'
COL_STUDENT_ANSWER = 'answer'
COL_CORRECT_ANSWER = 'correct_answer_label'

def generate_personalized_study_guide(csv_path, curriculum_path, answer_key_path):
    print(f"Loading student answers from: {csv_path}")
    print(f"Loading answer key from: {answer_key_path}")
    print(f"Loading curriculum dataset from: {curriculum_path}")

    # Load datasets
    try:
        student_answers = pd.read_csv(csv_path)
        answer_key = pd.read_csv(answer_key_path)
        cdl_curriculum = pd.read_excel(curriculum_path)
    except FileNotFoundError as e:
        return f"File not found error: {e}\nPlease check the paths and try again."

    # Check student answers against the answer key
    if COL_QUESTION_ID not in student_answers.columns or COL_QUESTION_ID not in answer_key.columns:
        return f"Error: Both CSVs must contain a '{COL_QUESTION_ID}' column to match them up."

    merged = pd.merge(student_answers, answer_key, on=COL_QUESTION_ID, how='left')

    if COL_STUDENT_ANSWER not in merged.columns or COL_CORRECT_ANSWER not in merged.columns:
        return f"Error: Check column names. Could not find '{COL_STUDENT_ANSWER}' or '{COL_CORRECT_ANSWER}' after merging."

    # Create a boolean correctness column by comparing strings (ignoring case and whitespace)
    merged['is_correct'] = (
        merged[COL_STUDENT_ANSWER].astype(str).str.strip().str.lower() ==
        merged[COL_CORRECT_ANSWER].astype(str).str.strip().str.lower()
    ).astype(int)

    # Filter questions for the study guide
    incorrect_answers = merged[merged['is_correct'] == 0]
    perfect_score = False

    if incorrect_answers.empty:
        print("Perfect score detected! Generating a comprehensive review guide based on all answered questions...")
        target_questions = merged
        perfect_score = True
    else:
        print(f"Found {len(incorrect_answers)} incorrect answers. Gathering context from Hybrid RAG...")
        target_questions = incorrect_answers

    # Gather context using our Hybrid RAG for the targeted topics/questions
    rag_context = ""
    for index, row in target_questions.iterrows():
        # Use the question ID or topic to search our RAG
        search_query = str(row.get('topic', f"Question ID {row.get(COL_QUESTION_ID)} concept"))

        # Fetch top 3 most relevant context blocks to expand the scope
        rag_results = hybrid_search(search_query, top_k=3)

        # Append the retrieved content (including Related Topics) to our context
        for _, q_row in rag_results.iterrows():
            rag_context += f"Topic: {q_row.get('Topic', 'Unknown')}\n"
            rag_context += f"Definition: {q_row.get('Definition', '')}\n"
            # Include related topics from the dataset so the LLM can expand the study guide
            rag_context += f"Related Topics to Mention: {q_row.get('Related Topics', 'None')}\n\n"

    # Inject context into LLM (OpenAI GPT API)
    print("Generating study guide via OpenAI API...")

    if perfect_score:
        prompt_intro = "A student has just taken a practice exam and got a perfect score! Provide an extremely comprehensive, advanced reviewer to reinforce and deepen their knowledge on these topics."
    else:
        prompt_intro = "A student has just taken a practice exam and got several topics wrong. Provide a personalized, extensively detailed reviewer to help them deeply understand their mistakes and master the material."

    prompt = f"""
    You are an expert Google Cloud Digital Leader tutor.
    {prompt_intro}

    CRITICAL INSTRUCTIONS:
    1. CLEAN PLAIN TEXT ONLY: DO NOT use any Markdown formatting. NO asterisks (*), NO hashtags (#), NO underscores (_), and NO bold/italic formatting. Use ALL CAPS for headers, simple numbers (1., 2.), and clean line breaks to structure your output.
    2. BE HIGHLY DESCRIPTIVE & CONTEXTUAL: Provide a massive, extensive, and comprehensive reviewer. Do not give brief summaries. Tie everything back to Google Cloud principles, business values, and real-world environments.
    3. MENTION RELATED TOPICS: In the context below, there are "Related Topics" listed for the main concepts. You MUST weave these related topics into your explanations to expand the length and depth of the study guide. Show how they connect!
    4. NO CRITIQUE: DO NOT critique the quiz itself or mention test format. Focus purely on teaching.

    For every topic the student needs to review, you MUST format the output strictly as follows:

    TOPIC: [TOPIC NAME IN ALL CAPS]

    DESCRIPTION AND IN-DEPTH EXPLANATION:
    (A comprehensive and highly detailed explanation of the concept and its contextual importance in Google Cloud.)

    RELATED CONCEPTS TO KNOW:
    (Explain the related topics provided in the context. Show how they interconnect with the main topic to build a broader understanding.)

    REAL-WORLD EXAMPLES:
    (Detailed, practical examples illustrating exactly how this concept is applied in a business or cloud environment.)

    GUIDE QUESTIONS:
    (3 to 5 thought-provoking questions to thoroughly test their understanding.)

    Context from Curriculum (Including Related Topics):
    {rag_context}

    Please provide the structured, extensive, plain-text reviewer now.
    """

    try:
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": "You are a highly professional, exhaustive, and helpful cloud tutor."},
                {"role": "user", "content": prompt}
            ],
            max_completion_tokens=4000
        )
        llm_response_content = response.choices[0].message.content
        print(f"LLM Response Content Length: {len(llm_response_content)}")
        print(f"LLM Response Content (first 500 chars):\n{llm_response_content[:500]}")
        return llm_response_content
    except Exception as e:
        return f"Error calling OpenAI API: {e}"

# EXECUTE THE PIPELINE

final_study_guide = generate_personalized_study_guide(STUDENT_ANSWERS_CSV_PATH, CDL_TOPIC_DATASET_PATH, QUIZ_ANSWER_KEY_PATH)
print("\n================ GENERATED STUDY GUIDE ================\n")
print(final_study_guide)
print("\n=======================================================\n")

Loading student answers from: /content/drive/MyDrive/Colab Notebooks/low_score_test.csv
Loading answer key from: /content/drive/MyDrive/Colab Notebooks/quiz_answer_key.csv
Loading curriculum dataset from: /content/drive/MyDrive/Colab Notebooks/cdl-TOPIC_DATASET.xlsx
Found 30 incorrect answers. Gathering context from Hybrid RAG...
Generating study guide via OpenAI API...
LLM Response Content Length: 9936
LLM Response Content (first 500 chars):
TOPIC: NEW VALUE VALUE

DESCRIPTION AND IN-DEPTH EXPLANATION:
"New value value" in the context of Google Cloud refers to the discovery of previously hidden benefits or insights within data through advanced data analysis and processing techniques. In today's corporate landscape, data is a pivotal asset. However, simply collecting data is not enough. The ability to extract meaningful insights from this data can critically transform business operations and strategies. Google Cloud provides an array

================ GENERATED STUDY GUIDE ============

Mid-Scores

In [ ]:
import pandas as pd

# File paths
STUDENT_ANSWERS_CSV_PATH = "/content/drive/MyDrive/Colab Notebooks/mid_score_test.csv"
CDL_TOPIC_DATASET_PATH = "/content/drive/MyDrive/Colab Notebooks/cdl-TOPIC_DATASET.xlsx"
QUIZ_ANSWER_KEY_PATH = "/content/drive/MyDrive/Colab Notebooks/quiz_answer_key.csv"

# Column name config
COL_QUESTION_ID = 'question_id'
COL_STUDENT_ANSWER = 'answer'
COL_CORRECT_ANSWER = 'correct_answer_label'

def generate_personalized_study_guide(csv_path, curriculum_path, answer_key_path):
    print(f"Loading student answers from: {csv_path}")
    print(f"Loading answer key from: {answer_key_path}")
    print(f"Loading curriculum dataset from: {curriculum_path}")

    # Load datasets
    try:
        student_answers = pd.read_csv(csv_path)
        answer_key = pd.read_csv(answer_key_path)
        cdl_curriculum = pd.read_excel(curriculum_path)
    except FileNotFoundError as e:
        return f"File not found error: {e}\nPlease check the paths and try again."

    # Check student answers against the answer key
    if COL_QUESTION_ID not in student_answers.columns or COL_QUESTION_ID not in answer_key.columns:
        return f"Error: Both CSVs must contain a '{COL_QUESTION_ID}' column to match them up."

    merged = pd.merge(student_answers, answer_key, on=COL_QUESTION_ID, how='left')

    if COL_STUDENT_ANSWER not in merged.columns or COL_CORRECT_ANSWER not in merged.columns:
        return f"Error: Check column names. Could not find '{COL_STUDENT_ANSWER}' or '{COL_CORRECT_ANSWER}' after merging."

    # Create a boolean correctness column by comparing strings (ignoring case and whitespace)
    merged['is_correct'] = (
        merged[COL_STUDENT_ANSWER].astype(str).str.strip().str.lower() ==
        merged[COL_CORRECT_ANSWER].astype(str).str.strip().str.lower()
    ).astype(int)

    # Filter questions for the study guide
    incorrect_answers = merged[merged['is_correct'] == 0]
    perfect_score = False

    if incorrect_answers.empty:
        print("Perfect score detected! Generating a comprehensive review guide based on all answered questions...")
        target_questions = merged
        perfect_score = True
    else:
        print(f"Found {len(incorrect_answers)} incorrect answers. Gathering context from Hybrid RAG...")
        target_questions = incorrect_answers

    # Gather context using our Hybrid RAG for the targeted topics/questions
    rag_context = ""
    for index, row in target_questions.iterrows():
        # Use the question ID or topic to search our RAG
        search_query = str(row.get('topic', f"Question ID {row.get(COL_QUESTION_ID)} concept"))

        # Fetch top 3 most relevant context blocks to expand the scope
        rag_results = hybrid_search(search_query, top_k=3)

        # Append the retrieved content (including Related Topics) to our context
        for _, q_row in rag_results.iterrows():
            rag_context += f"Topic: {q_row.get('Topic', 'Unknown')}\n"
            rag_context += f"Definition: {q_row.get('Definition', '')}\n"
            # Include related topics from the dataset so the LLM can expand the study guide
            rag_context += f"Related Topics to Mention: {q_row.get('Related Topics', 'None')}\n\n"

    # Inject context into LLM (OpenAI GPT API)
    print("Generating study guide via OpenAI API...")

    if perfect_score:
        prompt_intro = "A student has just taken a practice exam and got a perfect score! Provide an extremely comprehensive, advanced reviewer to reinforce and deepen their knowledge on these topics."
    else:
        prompt_intro = "A student has just taken a practice exam and got several topics wrong. Provide a personalized, extensively detailed reviewer to help them deeply understand their mistakes and master the material."

    prompt = f"""
    You are an expert Google Cloud Digital Leader tutor.
    {prompt_intro}

    CRITICAL INSTRUCTIONS:
    1. CLEAN PLAIN TEXT ONLY: DO NOT use any Markdown formatting. NO asterisks (*), NO hashtags (#), NO underscores (_), and NO bold/italic formatting. Use ALL CAPS for headers, simple numbers (1., 2.), and clean line breaks to structure your output.
    2. BE HIGHLY DESCRIPTIVE & CONTEXTUAL: Provide a massive, extensive, and comprehensive reviewer. Do not give brief summaries. Tie everything back to Google Cloud principles, business values, and real-world environments.
    3. MENTION RELATED TOPICS: In the context below, there are "Related Topics" listed for the main concepts. You MUST weave these related topics into your explanations to expand the length and depth of the study guide. Show how they connect!
    4. NO CRITIQUE: DO NOT critique the quiz itself or mention test format. Focus purely on teaching.

    For every topic the student needs to review, you MUST format the output strictly as follows:

    TOPIC: [TOPIC NAME IN ALL CAPS]

    DESCRIPTION AND IN-DEPTH EXPLANATION:
    (A comprehensive and highly detailed explanation of the concept and its contextual importance in Google Cloud.)

    RELATED CONCEPTS TO KNOW:
    (Explain the related topics provided in the context. Show how they interconnect with the main topic to build a broader understanding.)

    REAL-WORLD EXAMPLES:
    (Detailed, practical examples illustrating exactly how this concept is applied in a business or cloud environment.)

    GUIDE QUESTIONS:
    (3 to 5 thought-provoking questions to thoroughly test their understanding.)

    Context from Curriculum (Including Related Topics):
    {rag_context}

    Please provide the structured, extensive, plain-text reviewer now.
    """

    try:
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": "You are a highly professional, exhaustive, and helpful cloud tutor."},
                {"role": "user", "content": prompt}
            ],
            max_completion_tokens=4000
        )
        llm_response_content = response.choices[0].message.content
        print(f"LLM Response Content Length: {len(llm_response_content)}")
        print(f"LLM Response Content (first 500 chars):\n{llm_response_content[:500]}")
        return llm_response_content
    except Exception as e:
        return f"Error calling OpenAI API: {e}"

# EXECUTE THE PIPELINE

final_study_guide = generate_personalized_study_guide(STUDENT_ANSWERS_CSV_PATH, CDL_TOPIC_DATASET_PATH, QUIZ_ANSWER_KEY_PATH)
print("\n================ GENERATED STUDY GUIDE ================\n")
print(final_study_guide)
print("\n=======================================================\n")

Loading student answers from: /content/drive/MyDrive/Colab Notebooks/mid_score_test.csv
Loading answer key from: /content/drive/MyDrive/Colab Notebooks/quiz_answer_key.csv
Loading curriculum dataset from: /content/drive/MyDrive/Colab Notebooks/cdl-TOPIC_DATASET.xlsx
Found 15 incorrect answers. Gathering context from Hybrid RAG...
Generating study guide via OpenAI API...
LLM Response Content Length: 10684
LLM Response Content (first 500 chars):
TOPIC: NEW VALUE VALUE

DESCRIPTION AND IN-DEPTH EXPLANATION:
The concept of "New Value Value" in Google Cloud refers to the process of uncovering previously hidden benefits within data. Companies hold vast amounts of data, yet many fail to leverage it effectively to garner deeper insights that drive business growth and innovation. Google Cloud offers robust analytics and machine learning tools that enable organizations to sift through data silos, extract valuable insights, and translate them in

================ GENERATED STUDY GUIDE ===========

In [ ]:
import openai
from google.colab import userdata

OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
client = openai.OpenAI(api_key=OPENAI_API_KEY)

def ask_cdl_bot():
    print("Hello! I am your Google Cloud Digital Leader assistant. Ask me anything about the CDL curriculum.")
    print("Type 'exit' or 'quit' to end the conversation.")

    while True:
        user_query = input("\nAsk a question about the Google Cloud Digital Leader topics: ")

        if user_query.lower() in ['exit', 'quit']:
            print("Thank you for chatting! Goodbye.")
            break

        print("\nSearching for relevant context...")

        try:
            rag_results = hybrid_search(user_query, top_k=3)
        except Exception as e:
            print(f"Error during search: {e}")
            continue

        if rag_results.empty:
            print("No relevant context found in the CDL curriculum.")
            continue

        context_str = ""
        for index, row in rag_results.iterrows():
            context_str += f"Topic: {row.get('Topic', 'Unknown')}\n"
            context_str += f"Definition: {row.get('Definition', '')}\n"
            context_str += f"Related Topics: {row.get('Related Topics', 'None')}\n\n"

        prompt = f"""
        You are an expert Google Cloud Digital Leader tutor.
        Your task is to answer the user's question using the provided context as your factual foundation.
        However, DO NOT just repeat the context. You must generate a highly detailed, conversational, and comprehensive explanation of the concept using your advanced LLM capabilities.
        Provide real-world examples, elaborate on why the concept matters in cloud computing, and ensure your answer is rich and engaging (at least 2-3 paragraphs).

        CRITICAL INSTRUCTION: You must ONLY answer questions related to Google Cloud, cloud computing, and topics found in the provided CDL dataset context. If the user asks a question outside of these topics (e.g., general programming, math, history), politely refuse to answer and remind them that you are a Google Cloud Digital Leader tutor.

        Context from curriculum:
        {context_str}

        User Question: {user_query}
        """

        print("Generating explanation...")

        try:
            response = client.chat.completions.create(
                model="gpt-4o",
                messages=[
                    {"role": "system", "content": "You are a highly capable and conversational cloud tutor. You provide detailed explanations that go beyond just quoting facts. You ONLY answer questions related to Google Cloud and cloud computing."},
                    {"role": "user", "content": prompt}
                ],
                max_completion_tokens=800,
                temperature=0.7
            )
            answer = response.choices[0].message.content
            print("\n================ EXPLANATION ================\n")
            print(answer)
            print("\n=============================================\n")
        except Exception as e:
            print(f"Error calling OpenAI API: {e}")


ask_cdl_bot()

Hello! I am your Google Cloud Digital Leader assistant. Ask me anything about the CDL curriculum.
Type 'exit' or 'quit' to end the conversation.

Searching for relevant context...
Generating explanation...

================ EXPLANATION ================

Cloud service models are fundamental to understanding how cloud computing operates and are typically categorized into three primary types: Infrastructure as a Service (IaaS), Platform as a Service (PaaS), and Software as a Service (SaaS). Each model offers varying levels of control, flexibility, and management, catering to different business needs and technical requirements.

**Infrastructure as a Service (IaaS)** is the most flexible cloud model, providing virtualized computing resources over the internet. With IaaS, organizations can rent IT infrastructure—servers, storage, and networking—on a pay-as-you-go basis. This model is akin to managing your own data center but without the overhead of physical maintenance. IaaS is ideal for bu